# VBNL

This notebook generates the weekly Veteran By Name List (VBNL) workbook used by the veteran
homelessness workgroup for case conferencing.

Data is pulled from the [[ORGANIZATION_NAME]] Data Hub (Azure Blob Storage) and the Looker API, and includes:

- `episode_systemwide.parquet`: episode-level status, chronic status, CE flags, household composition
- `Client_Demographics.parquet`: veteran status, gender, race/ethnicity, current income (`MonthlyCashIncome`), disabling condition (`DisablingConditionAnytime`)
- **Looker**: client birth dates, VBNL form assessment data, enrollment data, CE assessments

The notebook produces a five-tab Excel workbook:

1. **Active** – veterans currently experiencing homelessness (`EpisodeOutflowType = Active`)
2. **Recently Inactive** – veterans who became inactive in the last 30 days
3. **No CE Engagement** – active veterans without an active CE enrollment
4. **Newly Assessed** – veterans with a Housing Triage Tool assessment in the last 8 days
5. **Flagged** – veterans flagged for re-review on the VBNL form

# Import Dependencies

Load required packages for data processing and Azure connections.


In [ ]:
import json
from datetime import datetime, timedelta
import looker_sdk
from looker_sdk import api_settings
from looker_sdk.sdk.api40 import models as models
import requests


import pandas as pd
import io
from azure.storage.blob import BlobServiceClient
from azure.identity import DefaultAzureCredential

In [ ]:
# #Comment out in production. Use for development

# Import Looker SDK connection from shared script
import sys
sys.path.insert(0, '..')
from [[LOCAL_SCRIPT_NAME]] import sdk

In [ ]:
# #Add chunk to connect to Looker API in production

# class [[ORG_PREFIX]]_Looker_API_Settings(api_settings.ApiSettings):
#     def __init__(self, *args, **kw_args):
#         self.my_var = kw_args.pop("my_var")
#         super().__init__(*args, **kw_args)

#     def read_config(self) -> api_settings.SettingsConfig:
#         config = super().read_config()
#         # See api_settings.SettingsConfig for required fields.
#         if self.my_var == "set":
#             config["base_url"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','LOOKER-API-BASE-URL')
#             config["client_id"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','LOOKER-API-CLIENT-ID')
#             config["client_secret"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','LOOKER-API-CLIENT-SECRET')          
#         return config

# sdk = looker_sdk.init40(config_settings=[[ORG_PREFIX]]_Looker_API_Settings(my_var="set"))

# vars(vars(vars(sdk)["transport"])["settings"])["timeout"] = 300

Run "az login" in the terminal before running this chunk; select production

In [ ]:
#In development outside of azure

# Define a helper function to get a table from '[[AZURE_CONTAINER_NAME]]'
def get_table(file_name):
    # Define the storage account and container name
    storage_account_name = "[[AZURE_STORAGE_ACCOUNT_NAME]]"
    container_name = "[[AZURE_CONTAINER_NAME]]"
    
    # Use DefaultAzureCredential to authenticate with Microsoft Entra credentials
    credential = DefaultAzureCredential()
    
    # Construct the BlobServiceClient URL with the storage account name
    blob_service_client = BlobServiceClient(
        account_url=f"https://{storage_account_name}.blob.core.windows.net",
        credential=credential
    )
    
    # Create a BlobClient for the specified blob
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=file_name)
    
    # Download the blob content into a stream
    download_stream = blob_client.download_blob()
    parquet_data = io.BytesIO(download_stream.readall())
    
    # Load the Parquet data into a Pandas DataFrame
    df = pd.read_parquet(parquet_data)
    
    # Return the dataframe
    return df

In [ ]:
# # For production in Azure

# # define a helper function to get a table from '[[AZURE_CONTAINER_NAME]]'
# def get_table(file_name):
#     # Define parameter fields
#     container_name = "[[AZURE_CONTAINER_NAME]]"

#     # Create a BlobServiceClient object using the connection string
#     connection_string = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]', '[[ADLS_CONNECTION_SECRET]]')
#     blob_service_client = BlobServiceClient.from_connection_string(connection_string)

#     # Create a BlobClient for the specified blob
#     blob_client = blob_service_client.get_blob_client(container=container_name, blob=file_name)

#     # Download the blob content into a stream
#     download_stream = blob_client.download_blob()
#     parquet_data = io.BytesIO(download_stream.readall())

#     # Load the Parquet data into a Pandas DataFrame
#     df = pd.read_parquet(parquet_data)

#     # Return the dataframe
#     return df

# Extract Data Tables from [[ORGANIZATION_NAME]] Data Hub

In [ ]:
# read in demographics first so we can build vet_ids for early filtering
demographics = get_table("Client_Demographics.parquet")
vet_ids = set(demographics.loc[demographics["VeteranStatus"] == "Yes", "ClientUniqueIdentifier"])

# read in and filter episodes to veterans only
episodes = get_table("episode_systemwide.parquet")
episodes = episodes[episodes["ClientUniqueIdentifier"].isin(vet_ids)].copy()

# read in events and filter to veterans only
# cast ClientUniqueIdentifier to str to avoid boolean dtype comparison warnings
events = get_table("event_systemwide.parquet")
events = events[events["ClientUniqueIdentifier"].astype(str).isin(vet_ids)].copy()
events["EventDate"] = pd.to_datetime(events["EventDate"], errors="coerce").dt.date

# read in and filter enrollments to veterans only
enrollments_all = get_table("All_Program_Enrollments.parquet")
if "ClientUniqueIdentifier" in enrollments_all.columns:
    enrollments_all = enrollments_all[enrollments_all["ClientUniqueIdentifier"].isin(vet_ids)].copy()
enrollments_all["EnrollmentID"] = enrollments_all["EnrollmentID"].astype(str)

# read in program attributes (for agency/program name lookup) — not client-level, no filtering needed
program_attributes = get_table("Program_Attributes.parquet")

# Looker Query Helper

In [ ]:
# Helper function to retry Looker queries on timeout with exponential backoff
def run_query_with_retry(sdk, body, result_format="json", max_retries=5, base_delay=15):
    """
    Run a Looker inline query with retry logic for timeouts.

    Args:
        sdk: Looker SDK instance
        body: Query body dictionary (model, view, fields, filters, limit)
        result_format: Output format (default: "json")
        max_retries: Maximum number of retry attempts (default: 5)
        base_delay: Base delay in seconds between retries (default: 15)

    Returns:
        Query result as a string

    Raises:
        Exception: If all retries are exhausted
    """
    import time
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return sdk.run_inline_query(result_format=result_format, body=body)
        except Exception as e:
            last_error = e
            error_str = str(e).lower()
            # Check if it's a timeout or server error worth retrying
            if "timeout" in error_str or "504" in str(e) or "503" in str(e) or "502" in str(e) or "timed out" in error_str:
                if attempt < max_retries:
                    wait_time = base_delay * (2 ** attempt)  # Exponential backoff: 15, 30, 60, 120, 240
                    print(f"Query timed out. Retrying in {wait_time} seconds... (attempt {attempt + 1}/{max_retries})")
                    time.sleep(wait_time)
                    continue
            raise  # Re-raise if not a retryable error
    raise last_error

# Looker Data Pulls

In [ ]:
# ── 1. Client birth dates, names, and SSN ────────────────────────────────────────────
result = run_query_with_retry(sdk, {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "client_model",
    "fields": [
        "client_model.unique_identifier",
        "client_model.birth_date",
        "client_model.first_name",
        "client_model.last_name",
        "client_model.ssn",          # full SSN (for Active/Inactive tabs)
        "client_model.ssn3",         # SSN Last 4 (for Newly Assessed tab)
    ],
    "filters": {"client_model.deleted": "No"},
    "limit": -1,
})
client_info = pd.DataFrame(json.loads(result))

# ── 2. VBNL form data (most recent per client) ────────────────────────────────────────
# Assessment: [[VBNL_FORM_ASSESSMENT_NAME]] (ref_assessment=[[VBNL_FORM_ASSESSMENT_ID]])
result = run_query_with_retry(sdk, {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "client_model",
    "fields": [
        "clients.unique_identifier",
        "client_assessments.assessment_date",
        "client_assessment_custom.c_kc_volt_agency_assigned",                  # Routed To
        "client_assessment_custom.c_kc_volt_housing_plan",                     # Housing Plan Detail
        "client_assessment_custom.c_reg_sex_offender",                         # Registered Sex Offender
        "client_assessment_custom.c_kc_volt_eligible__va_funded_programs",     # Eligible for VA Funded Housing Programs
        "client_assessment_custom.c_kc_volt_notes",                            # VBNL Note
        "client_assessment_custom.c_kc_volt_review_flag",                      # Flag for Re-Review
    ],
    "filters": {
        "client_model.deleted":              "No",
        "static_demographics.veteran_text":  "Yes",
        "client_assessments.ref_assessment": "[[VBNL_FORM_ASSESSMENT_ID]]",   # [[VBNL_FORM_ASSESSMENT_NAME]]
        "client_assessments.deleted":        "No",
    },
    "limit": -1,
})
vbnl_form = (
    pd.DataFrame(json.loads(result))
    .sort_values("client_assessments.assessment_date", ascending=False)
    .drop_duplicates(subset="clients.unique_identifier", keep="first")
)

# ── Shared: entry screen / health data for all veteran clients ────────────────────────
# Pulled once here and reused for both Newly Assessed (query 3) and Flagged (query 4).
# client_model enrollments join independently from client_assessments, so combining
# entry_screen fields with assessment fields creates a cross join. Both queries pull
# assessment data separately and match entry screens by date proximity in Python.
result = run_query_with_retry(sdk, {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "client_model",
    "fields": [
        "clients.unique_identifier",
        "entry_screen.added_date",
        "household_entry_screen.total_household_income",
        "entry_custom.c_kc_is_household_chronically_homeless",
        "chronically_homeless_households.is_chronic_homeless_household",
        "entry_screen.health_chronic",
        "entry_screen.health_chronic_longterm",
        "entry_screen.health_dev_disability",
        "entry_screen.health_mental",
        "entry_screen.health_mental_longterm",
        "entry_screen.health_phys_disability",
        "entry_screen.health_phys_disability_longterm",
        "entry_screen.health_substance_abuse",
        "entry_screen.health_substance_abuse_longterm",
        "entry_screen.health_insurance",
        "entry_screen.income_vet_disability_is",
        "entry_screen.income_vet_pension_is",
        "entry_screen.income_earned",
        "entry_screen.benefits_noncash",
    ],
    "filters": {
        "client_model.deleted":             "No",
        "static_demographics.veteran_text": "Yes",
        "enrollments.date_filter":          "NULL",
    },
    "limit": -1,
})
_all_entry_screens = pd.DataFrame(json.loads(result))
_all_entry_screens["entry_screen.added_date"] = pd.to_datetime(
    _all_entry_screens["entry_screen.added_date"], errors="coerce"
)

def _match_entry_screens(base_df, assessment_date_col):
    """
    For each client in base_df, find the most recent entry screen in _all_entry_screens
    where entry_screen.added_date <= the client's assessment date. Returns a merged df.
    """
    _dates = base_df[["clients.unique_identifier", assessment_date_col]].copy()
    _dates[assessment_date_col] = pd.to_datetime(_dates[assessment_date_col], errors="coerce")
    return (
        _dates
        .merge(_all_entry_screens, on="clients.unique_identifier", how="left")
        .loc[lambda d: d["entry_screen.added_date"] <= d[assessment_date_col]]
        .sort_values("entry_screen.added_date", ascending=False)
        .drop_duplicates(subset="clients.unique_identifier", keep="first")
        .drop(columns=assessment_date_col)
    )

# ── shared demo/disability lookup ─────────────────────────────────────────────────────
_demo_dis = (
    demographics[["ClientUniqueIdentifier", "DisablingConditionAnytime"]]
    .rename(columns={"ClientUniqueIdentifier": "clients.unique_identifier"})
)

# ── 3. Newly Assessed ─────────────────────────────────────────────────────────────────
# Veterans with a CE assessment in the last 8 days.
# Uses client_model (not client view) to include assessments outside of CESP.
# [[CE_ASSESSMENT_IDS]] = your community's Coordinated Entry assessment type(s)
# (e.g., a current Housing Triage Tool plus any legacy/prior versions)
# Entry screen fields are matched separately by date to avoid cross join.
eight_days_ago = (datetime.today() - timedelta(days=8)).strftime("%Y/%m/%d")
result = run_query_with_retry(sdk, {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "client_model",
    "fields": [
        "clients.unique_identifier",
        "client_assessments.id",
        "client_assessments.assessment_date",
        "client_model.birth_date",
        "client_assessments.ref_agency_text",                                   # Assessing Agency
        "screens.name",
        "client_assessments.ref_user",                                          # User Creating
        "client_model.first_name",
        "client_model.last_name",
        "client_model.ssn3",
        "static_demographics.gender_text",
        "client_assessments.ssvf_targeting_field_12",                           # Registered Sex Offender
    ],
    "filters": {
        "client_model.deleted":              "No",
        "static_demographics.veteran_text":  "Yes",
        "client_assessments.assessment_date": f"after {eight_days_ago}",
        "client_assessments.ref_assessment":  "[[CE_ASSESSMENT_IDS]]",
        "client_assessments.deleted":         "No",
    },
    "limit": -1,
})
newly_assessed_raw = (
    pd.DataFrame(json.loads(result))
    .assign(**{
        "client.birth_date": lambda d: d["client_model.birth_date"],
        "client.ssn3":       lambda d: d["client_model.ssn3"],
        "client.full_name":  lambda d: d["client_model.first_name"].fillna("") + " " + d["client_model.last_name"].fillna(""),
        "agencies.name":     lambda d: d["client_assessments.ref_agency_text"],
    })
    .sort_values("client_assessments.assessment_date", ascending=False)
    .drop_duplicates(subset="clients.unique_identifier", keep="first")
    .merge(_demo_dis, on="clients.unique_identifier", how="left")
)
newly_assessed_raw = newly_assessed_raw.merge(
    _match_entry_screens(newly_assessed_raw, "client_assessments.assessment_date"),
    on="clients.unique_identifier", how="left"
)
newly_assessed_raw = newly_assessed_raw[
    pd.to_datetime(newly_assessed_raw["client_assessments.assessment_date"], errors="coerce").dt.date <= datetime.today().date()
]

# ── 4. Flagged for Re-Review ──────────────────────────────────────────────────────────
# Assessment: [[VBNL_FORM_ASSESSMENT_NAME]] (ref_assessment=[[VBNL_FORM_ASSESSMENT_ID]])
#
# The flag filter is intentionally NOT applied in Looker — see comment below.
# Entry screen fields matched separately by date (reuses _all_entry_screens).

# ── 4a. VBNL assessment data (no entry_screen fields) ────────────────────────────────
_flagged_result = run_query_with_retry(sdk, {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "client_model",
    "fields": [
        "clients.unique_identifier",
        "client_assessments.id",
        "client_assessments.assessment_date",
        "client_model.birth_date",
        "client_assessments.ref_agency_text",                                   # Assessing Agency
        "screens.name",
        "client_assessments.ref_user",                                          # User Creating
        "client_model.first_name",
        "client_model.last_name",
        "client_model.ssn3",
        "static_demographics.gender_text",
        "client_assessment_custom.c_reg_sex_offender",                          # Registered Sex Offender (VBNL field)
        "client_assessment_custom.c_kc_volt_notes",                             # VBNL Notes
        "client_assessment_custom.c_kc_volt_review_flag",                       # Flag for Re-Review
    ],
    "filters": {
        "client_model.deleted":              "No",
        "static_demographics.veteran_text":  "Yes",
        "client_assessments.ref_assessment": "[[VBNL_FORM_ASSESSMENT_ID]]",   # [[VBNL_FORM_ASSESSMENT_NAME]]
        "client_assessments.deleted":        "No",
    },
    "limit": -1,
})
flagged_raw = (
    pd.DataFrame(json.loads(_flagged_result))
    .assign(**{
        "client.birth_date":  lambda d: d["client_model.birth_date"],
        "client.ssn3":        lambda d: d["client_model.ssn3"],
        "client.full_name":   lambda d: d["client_model.first_name"].fillna("") + " " + d["client_model.last_name"].fillna(""),
        "agencies.name":      lambda d: d["client_assessments.ref_agency_text"],
    })
    # Keep only VBNL form rows (screens.name confirms ref_assessment resolved correctly)
    .loc[lambda d: d["screens.name"] == "[[VBNL_FORM_ASSESSMENT_NAME]]"]
    # Latest VBNL assessment per client: MAX date, MAX id as tiebreaker (matches SQL CTE)
    .sort_values(
        ["client_assessments.assessment_date", "client_assessments.id"],
        ascending=[False, False],
    )
    .drop_duplicates(subset="clients.unique_identifier", keep="first")
    # Only clients whose latest VBNL assessment is currently flagged
    .loc[lambda d: d["client_assessment_custom.c_kc_volt_review_flag"] == "Yes"]
    .merge(_demo_dis, on="clients.unique_identifier", how="left")
)
flagged_raw = flagged_raw.merge(
    _match_entry_screens(flagged_raw, "client_assessments.assessment_date"),
    on="clients.unique_identifier", how="left"
)

# ── 5. SQUARES on File ────────────────────────────────────────────────────────────────
# Clients who have a file attachment named "SQUARES" on their record.
# Used to derive a Yes/No indicator merged into base.
result = run_query_with_retry(sdk, {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "client_model",
    "fields": ["clients.unique_identifier"],
    "filters": {
        "client_model.deleted":             "No",
        "static_demographics.veteran_text": "Yes",
        "client_file_names.name":           "SQUARES",
    },
    "limit": -1,
})
squares_ids = set(
    pd.DataFrame(json.loads(result))["clients.unique_identifier"].astype(str)
)

# ── 6. Active CE enrollment (live) ───────────────────────────────────────────────────
# Replaces CEActive from episode_systemwide.parquet to avoid stale snapshot issues.
# A veteran is CE-active if they have an open enrollment in your community's Coordinated Entry project (ProgramID [[CE_PROGRAM_ID]]).
# date_filter NULL disables the base explore's default 90-day window so enrollments
# that started >90 days ago but are still open are included.
result = run_query_with_retry(sdk, {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "base",
    "fields": ["clients.unique_identifier"],
    "filters": {
        "static_demographics.veteran_text": "Yes",
        "programs.id":                      "[[CE_PROGRAM_ID]]",
        "enrollments.end_date":             "NULL",
        "enrollments.date_filter":          "NULL",
    },
    "limit": "-1",
})
ce_active_ids = set(
    pd.DataFrame(json.loads(result))["clients.unique_identifier"].astype(str)
)

In [ ]:
# ── Debug: Flagged tab discrepancy investigation ──────────────────────────────────────
flagged_all = pd.DataFrame(json.loads(_flagged_result))

print("=== Row counts ===")
print(f"Raw rows from Looker:            {len(flagged_all)}")
print(f"Unique clients (raw):            {flagged_all['clients.unique_identifier'].nunique()}")

_after_dedup = (
    flagged_all
    .sort_values(["client_assessments.assessment_date", "client_assessments.id"], ascending=[False, False])
    .drop_duplicates(subset="clients.unique_identifier", keep="first")
)
print(f"After dedup (latest per client): {len(_after_dedup)}")

print(f"\n=== Flag column values (after dedup) ===")
print(_after_dedup["client_assessment_custom.c_kc_volt_review_flag"].value_counts(dropna=False))

_after_flag = _after_dedup.loc[_after_dedup["client_assessment_custom.c_kc_volt_review_flag"] == "Yes"]
print(f"\nAfter flag == 'Yes' filter:      {len(_after_flag)}")

print(f"\n=== Columns returned ===")
print(flagged_all.columns.tolist())

In [ ]:
# ── QA: Assessment type name/ID lookup ───────────────────────────────────────────────
# Pull all distinct assessment names and their IDs to verify ref_assessment values.
_qa_screens = run_query_with_retry(sdk, {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "client",
    "fields": [
        "screens.id",
        "screens.name",
    ],
    "filters": {},
    "limit": "-1",
})
qa_screens = (
    pd.DataFrame(json.loads(_qa_screens))
    .drop_duplicates()
    .sort_values("screens.name")
)
print(f"Columns returned: {qa_screens.columns.tolist()}")
print(f"Total distinct assessment types: {len(qa_screens)}")
qa_screens

In [ ]:
# ── 7. Enrolling user info for most recent homeless enrollment ────────────────────────
# For the Active tab: who enrolled the veteran into the homeless program that started
# their current episode. Matched later against EpisodeStartDate.
result = run_query_with_retry(sdk, {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "client_model",
    "fields": [
        "client_model.unique_identifier",
        "enrollments.id",
        "enrollments.ref_user",             # enrolling user name (join key for email lookup)
        "enrollments.ref_user_home_agency", # enrolling user's home agency
    ],
    "filters": {
        "client_model.deleted":             "No",
        "static_demographics.veteran_text": "Yes",
        "enrollments.date_filter":          "NULL",  # disable default 90-day window
    },
    "limit": -1,
})
enroll_user_info = (
    pd.DataFrame(json.loads(result))
    .rename(columns={
        "client_model.unique_identifier":       "ClientUniqueIdentifier",
        "enrollments.id":                       "EnrollmentID",
        "enrollments.ref_user":                 "EnrollingUserName",
        "enrollments.ref_user_home_agency":     "EnrollingUserAgency",
    })
)

# ── 7a. User email lookup (Project Descriptor model) ─────────────────────────────────
# members.name matches enrollments.ref_user — join to get email addresses for enrolling staff
result = run_query_with_retry(sdk, {
    "model": "[[LOOKER_MODEL_NAME]]",
    "view": "agencies",
    "fields": [
        "members.name",       # username — same value as enrollments.ref_user (join key)
        "members.full_name",  # CONCAT(first_name, ' ', last_name)
        "members.email",
    ],
    "filters": {
        "members.deleted": "No",
    },
    "limit": -1,
})
user_email_lookup = (
    pd.DataFrame(json.loads(result))
    .rename(columns={
        "members.name":       "EnrollingUserName",
        "members.full_name":  "EnrollingUserFullName",
        "members.email":      "EnrollingUserEmail",
    })
    .drop_duplicates("EnrollingUserName")
)

enroll_user_info = enroll_user_info.merge(user_email_lookup, on="EnrollingUserName", how="left")

# Prepare Base Data

In [ ]:
today = datetime.today().date()
run_date = datetime.today().strftime("%Y-%m-%d")
one_week_ago = (datetime.today() - timedelta(days=7)).date()
sixty_days_ago = (datetime.today() - timedelta(days=60)).date()
ninety_days_ago = (datetime.today() - timedelta(days=90)).date()
will_be_inactive_cutoff = today - timedelta(days=23)

# ── Deduplicate episodes: most recent TimePeriodStartDate per client ──────────────────
ep_vets = (
    episodes
    .sort_values("EpisodeStartDate", ascending=False)
    .drop_duplicates(subset="ClientUniqueIdentifier", keep="first")
    .copy()
)

# ── First Identification Date: earliest EpisodeStartDate ever per client ──────────────
first_id = (
    episodes
    .groupby("ClientUniqueIdentifier", as_index=False)["EpisodeStartDate"]
    .min()
    .rename(columns={"EpisodeStartDate": "FirstIdentificationDate"})
)

# ── Household size ────────────────────────────────────────────────────────────────────
ep_vets["HouseholdSize"] = (
    ep_vets["EpisodeMaxCountAdults"].fillna(0) + ep_vets["EpisodeMaxCountChildren"].fillna(0)
).astype(int)

# ── Normalize boolean-like fields to Yes/No ───────────────────────────────────────────
def to_yes_no(val):
    if val is True or val == 1 or str(val).strip().lower() in ("yes", "true", "1"):
        return "Yes"
    return "No"

ep_vets["ChronicallyHomeless"] = ep_vets["EpisodeChronicStatus"].apply(to_yes_no)

# ── Active CE enrollment — live Looker query, not parquet snapshot ────────────────────
ep_vets["ActiveCEEnrollment"] = (
    ep_vets["ClientUniqueIdentifier"].astype(str).isin(ce_active_ids).map({True: "Yes", False: "No"})
)

# ── Will Be Inactive / Needs Homelessness Status Confirmed ───────────────────────────
# Most recent Homeless ClientStatus event >= 23 days old means the episode will likely
# end before the next weekly list runs (episodes close after 30 days of no events).
will_be_inactive = (
    events[events["ClientStatus"] == "Homeless"]
    .sort_values("EventDate", ascending=False)
    .drop_duplicates(subset="ClientUniqueIdentifier", keep="first")
    [["ClientUniqueIdentifier", "EventDate"]]
)
will_be_inactive["WillBeInactive"] = will_be_inactive["EventDate"].apply(
    lambda d: "Yes" if pd.notna(d) and d <= will_be_inactive_cutoff else "No"
)

# ── Enrollment data (from parquet) ────────────────────────────────────────────────────
# If enrollments_all already has ClientUniqueIdentifier, use it directly;
# otherwise join through demographics via PersonalID.
if "ClientUniqueIdentifier" in enrollments_all.columns:
    enroll_with_uid = enrollments_all
else:
    personal_id_map = demographics[["PersonalID", "ClientUniqueIdentifier"]].drop_duplicates("PersonalID")
    enroll_with_uid = enrollments_all.merge(personal_id_map, on="PersonalID", how="inner")

# Normalize ProgramID to string in both tables to avoid int64/object merge error
enroll_with_uid = enroll_with_uid.copy()
enroll_with_uid["ProgramID"] = enroll_with_uid["ProgramID"].astype(str)
prog_attrs = program_attributes.copy()
prog_attrs["ProgramID"] = prog_attrs["ProgramID"].astype(str)

_open_enrollments = (
    enroll_with_uid
    .loc[lambda df: df["ClientUniqueIdentifier"].isin(vet_ids)]
    .loc[lambda df: df["ProjectExitDate"].isna()]
    .sort_values("ProjectStartDate", ascending=False)
    [["ClientUniqueIdentifier", "ProjectStartDate", "ProgramID"]]
    .merge(
        prog_attrs[["ProgramID", "ProgramName", "AgencyName", "ProjectTypeCode"]],
        on="ProgramID", how="left"
    )
)

# Most recent open enrollment per veteran — used for No CE Engagement tab header row
enrollments_vet_latest = (
    _open_enrollments
    .drop_duplicates(subset="ClientUniqueIdentifier", keep="first")
)

# All open enrollments per veteran — used for No CE Engagement tab (one row per enrollment)
enrollments_vet_all_open = _open_enrollments.copy()

# ── Household income: parquet-based, summed across open household members ─────────────
_vet_open_enroll_hh = (
    enroll_with_uid
    .loc[lambda df: df["ClientUniqueIdentifier"].isin(vet_ids)]
    .loc[lambda df: df["ProjectExitDate"].isna()]
    .sort_values("ProjectStartDate", ascending=False)
    .drop_duplicates(subset="ClientUniqueIdentifier", keep="first")
    [["ClientUniqueIdentifier", "HouseholdID"]]
)
_open_hh_members = (
    enroll_with_uid
    .loc[lambda df: df["ProjectExitDate"].isna()]
    [["ClientUniqueIdentifier", "HouseholdID"]]
    .drop_duplicates()
)
hh_income = (
    _vet_open_enroll_hh
    .rename(columns={"ClientUniqueIdentifier": "VetUID"})
    .merge(_open_hh_members, on="HouseholdID", how="left")
    .merge(
        demographics[["ClientUniqueIdentifier", "MonthlyCashIncome"]],
        on="ClientUniqueIdentifier", how="left"
    )
    .groupby("VetUID", as_index=False)["MonthlyCashIncome"]
    .sum()
    .rename(columns={"VetUID": "ClientUniqueIdentifier", "MonthlyCashIncome": "HouseholdTotalIncome"})
)

# ── Join all sources into base ────────────────────────────────────────────────────────
base = (
    ep_vets
    .merge(first_id, on="ClientUniqueIdentifier", how="left")
    .merge(
        demographics[["ClientUniqueIdentifier", "Gender", "RaceAndEthnicity", "MonthlyCashIncome"]]
            .rename(columns={"MonthlyCashIncome": "IncomeMostCurrent"})
            .drop_duplicates("ClientUniqueIdentifier"),
        on="ClientUniqueIdentifier", how="left"
    )
    .merge(
        client_info
            .rename(columns={
                "client_model.unique_identifier": "ClientUniqueIdentifier",
                "client_model.birth_date":        "_BirthDate",
                "client_model.first_name":        "FirstName",
                "client_model.last_name":         "LastName",
                "client_model.ssn":               "SSN",
                "client_model.ssn3":              "SSNLast4",
            })
            .drop_duplicates("ClientUniqueIdentifier"),
        on="ClientUniqueIdentifier", how="left"
    )
    .merge(
        vbnl_form
            .rename(columns={
                "clients.unique_identifier":          "ClientUniqueIdentifier",
                "client_assessments.assessment_date": "VBNLFormLastUpdateDate",
            })
            .drop_duplicates("ClientUniqueIdentifier"),
        on="ClientUniqueIdentifier", how="left"
    )
    .merge(
        will_be_inactive[["ClientUniqueIdentifier", "WillBeInactive"]],
        on="ClientUniqueIdentifier", how="left"
    )
    .merge(hh_income, on="ClientUniqueIdentifier", how="left")
)

# Veterans with no Homeless events at all get "No"
base["WillBeInactive"] = base["WillBeInactive"].fillna("No")

# ── SQUARES on File ───────────────────────────────────────────────────────────────────
base["SQUARESOnFile"] = base["ClientUniqueIdentifier"].astype(str).isin(squares_ids).map({True: "Yes", False: "No"})

# ── Compute age ───────────────────────────────────────────────────────────────────────
base["Age"] = pd.to_datetime(base["_BirthDate"], errors="coerce").apply(
    lambda dob: int((pd.Timestamp(today) - dob).days // 365) if pd.notna(dob) else pd.NA
)

# Build Workbook Tabs

In [ ]:
def select_columns(df, col_map):
    """Build output DataFrame from df using col_map {source_col: output_name}.
    Source columns prefixed with '_todo_' or otherwise absent in df are filled with pd.NA.
    Column order in the output matches the order of col_map.
    """
    result = pd.DataFrame()
    for src, dst in col_map.items():
        result[dst] = df[src].values if src in df.columns else pd.NA
    return result


# ── Tab 1: Active ──────────────────────────────────────────────────────────────────────
ACTIVE_COL_MAP = {
    "ClientUniqueIdentifier":                                                   "Unique Identifier",
    "LastName":                                                                 "Last Name",
    "FirstName":                                                                "First Name",
    "SSN":                                                                      "SSN",
    "Age":                                                                      "Age",
    "RaceAndEthnicity":                                                         "Race/Ethnicity",
    "Gender":                                                                   "Gender",
    "LastShelterStatusInTimeframe":                                             "Status",
    "FirstIdentificationDate":                                                  "First Identification Date",
    "EpisodeStartDate":                                                         "Current Episode Start Date",
    "EpisodeEndDate":                                                           "Current Episode End Date",
    "VBNLFormLastUpdateDate":                                                   "VBNL Form Last Update Date",
    "HouseholdSize":                                                            "Household Size",
    "HouseholdTotalIncome":                                                     "Total Household Income",
    "ChronicallyHomeless":                                                      "Chronically Homeless",
    "client_assessment_custom.c_kc_volt_agency_assigned":                      "Routed To",
    "client_assessment_custom.c_kc_volt_housing_plan":                         "Housing Plan Detail",
    "client_assessment_custom.c_reg_sex_offender":                             "Registered Sex Offender",
    "client_assessment_custom.c_kc_volt_eligible__va_funded_programs":         "Eligible for VA Funded Housing Programs",
    "client_assessment_custom.c_kc_volt_notes":                                "VBNL Note",
    "ActiveCEEnrollment":                                                       "Active CE Enrollment",
    "SQUARESOnFile":                                                            "SQUARES on File",
    "WillBeInactive":                                                           "Needs Homelessness Status Confirmed",
    "MostRecentActivity":                                                       "Most Recent Activity",
    "EnrollingUserAgency":                                                      "Enrolling User Home Agency",
    "EnrollingUserFullName":                                                    "Enrolling User Name",
    "EnrollingUserEmail":                                                       "Enrolling User Email Address",
}

active_df = base[
    (base["EpisodeOutflowType"] == "Active") |
    (
        (base["EpisodeOutflowType"] == "Inactive") &
        (pd.to_datetime(base["EpisodeEndDate"], errors="coerce").dt.date >= sixty_days_ago)
    )
].copy()
active_df.loc[active_df["EpisodeOutflowType"] == "Active", "EpisodeEndDate"] = pd.NaT
active_df["_end_sort"] = pd.to_datetime(active_df["EpisodeEndDate"], errors="coerce")
active_df = active_df.sort_values("_end_sort", ascending=False, na_position="first").drop(columns="_end_sort")
n_active_vets = int((active_df["EpisodeOutflowType"] == "Active").sum())
n_recently_inactive_vets = int((active_df["EpisodeOutflowType"] == "Inactive").sum())
active_ids = set(active_df["ClientUniqueIdentifier"])
# Row numbers in the Active sheet where recently-inactive people appear (1-based, +1 for header)
inactive_excel_rows = [
    i + 2
    for i, v in enumerate(active_df["EpisodeOutflowType"] == "Inactive")
    if v
]

# ── Attach enrolling-user info (Active tab only) ──────────────────────────────────────
# Use the EnrollmentID from each vet's most recent event to find the relevant enrollment.
_most_recent_event_enrollment = (
    events[events["ClientUniqueIdentifier"].isin(active_ids)]
    .sort_values("EventDate", ascending=False)
    .drop_duplicates(subset="ClientUniqueIdentifier", keep="first")
    [["ClientUniqueIdentifier", "EnrollmentID", "EventType", "EventDate"]]
    .assign(
        EnrollmentID=lambda d: d["EnrollmentID"].astype(str),
        MostRecentActivity=lambda d: d["EventType"] + " - " + d["EventDate"].astype(str),
    )
)
_user_info_matched = (
    enroll_user_info
    .assign(EnrollmentID=lambda d: d["EnrollmentID"].astype(str))
    .drop(columns="ClientUniqueIdentifier")  # avoid collision — _most_recent_event_enrollment has it
    .merge(_most_recent_event_enrollment, on="EnrollmentID", how="inner")
    .drop_duplicates("ClientUniqueIdentifier", keep="first")
    [["ClientUniqueIdentifier", "MostRecentActivity", "EnrollingUserFullName", "EnrollingUserName", "EnrollingUserEmail", "EnrollingUserAgency"]]
)
active_df = active_df.merge(_user_info_matched, on="ClientUniqueIdentifier", how="left")

tab_active = select_columns(active_df, ACTIVE_COL_MAP)


# ── Tab 2: Inactive Since Last BNL ────────────────────────────────────────────────────
# Exclude anyone who also has an active episode — they've already re-entered and
# belong on the Active tab, not here.
INACTIVE_COL_MAP = {
    "ClientUniqueIdentifier":                                                   "Unique Identifier",
    "LastName":                                                                 "Last Name",
    "FirstName":                                                                "First Name",
    "SSN":                                                                      "SSN",
    "Age":                                                                      "Age",
    "RaceAndEthnicity":                                                         "Race/Ethnicity",
    "Gender":                                                                   "Gender",
    "LastShelterStatusInTimeframe":                                             "Status",
    "FirstIdentificationDate":                                                  "First Identification Date",
    "EpisodeEndDate":                                                           "Inactive Date",
    "VBNLFormLastUpdateDate":                                                   "VBNL Form Last Update Date",
    "HouseholdSize":                                                            "Household Size",
    "HouseholdTotalIncome":                                                     "Total Household Income",
    "ChronicallyHomeless":                                                      "Chronically Homeless",
    "client_assessment_custom.c_kc_volt_agency_assigned":                      "Routed To",
    "client_assessment_custom.c_kc_volt_housing_plan":                         "Housing Plan Detail",
    "client_assessment_custom.c_reg_sex_offender":                             "Registered Sex Offender",
    "client_assessment_custom.c_kc_volt_eligible__va_funded_programs":         "Eligible for VA Funded Housing Programs",
    "client_assessment_custom.c_kc_volt_notes":                                "VBNL Note",
    "ActiveCEEnrollment":                                                       "Active CE Enrollment",
    "SQUARESOnFile":                                                            "SQUARES on File",
    "WillBeInactive":                                                           "Needs Homelessness Status Confirmed",
}

inactive_df = base[
    (base["EpisodeOutflowType"] == "Inactive") &
    (pd.to_datetime(base["EpisodeEndDate"], errors="coerce").dt.date >= ninety_days_ago) &
    (pd.to_datetime(base["EpisodeEndDate"], errors="coerce").dt.date < sixty_days_ago) &
    (~base["ClientUniqueIdentifier"].isin(active_ids))
].copy()
tab_inactive = select_columns(inactive_df, INACTIVE_COL_MAP)


# ── Tab 3: No CE Engagement ────────────────────────────────────────────────────────────
# Active veterans with no active CE enrollment; all open enrollments shown (one row per enrollment)
no_ce_base = active_df[active_df["ActiveCEEnrollment"] == "No"].copy()

no_ce_df = no_ce_base.merge(
    enrollments_vet_all_open,
    on="ClientUniqueIdentifier", how="left"
)

# Fall back to enrollment from most recent homeless event for vets with no open enrollments
_no_enroll_ids = no_ce_df.loc[no_ce_df["AgencyName"].isna(), "ClientUniqueIdentifier"].unique()
if len(_no_enroll_ids):
    _fallback_enroll = (
        events[
            events["ClientUniqueIdentifier"].isin(_no_enroll_ids) &
            (events["ClientStatus"] == "Homeless")
        ]
        .sort_values("EventDate", ascending=False)
        .drop_duplicates(subset="ClientUniqueIdentifier", keep="first")
        .assign(EnrollmentID=lambda d: d["EnrollmentID"].astype(str))
        [["ClientUniqueIdentifier", "EnrollmentID"]]
        .merge(
            enroll_with_uid[["EnrollmentID", "ProgramID", "ProjectStartDate"]],
            on="EnrollmentID", how="left"
        )
        .merge(
            prog_attrs[["ProgramID", "ProgramName", "AgencyName", "ProjectTypeCode"]],
            on="ProgramID", how="left"
        )
        [["ClientUniqueIdentifier", "AgencyName", "ProgramName", "ProjectTypeCode", "ProjectStartDate"]]
        .rename(columns={
            "AgencyName":       "_fb_AgencyName",
            "ProgramName":      "_fb_ProgramName",
            "ProjectTypeCode":  "_fb_ProjectTypeCode",
            "ProjectStartDate": "_fb_ProjectStartDate",
        })
    )
    no_ce_df = no_ce_df.merge(_fallback_enroll, on="ClientUniqueIdentifier", how="left")
    for col, fb in [
        ("AgencyName",      "_fb_AgencyName"),
        ("ProgramName",     "_fb_ProgramName"),
        ("ProjectTypeCode", "_fb_ProjectTypeCode"),
        ("ProjectStartDate","_fb_ProjectStartDate"),
    ]:
        no_ce_df[col] = no_ce_df[col].fillna(no_ce_df[fb])
    no_ce_df = no_ce_df.drop(columns=[c for c in no_ce_df.columns if c.startswith("_fb_")])

NO_CE_COL_MAP = {
    "ClientUniqueIdentifier":  "Unique Identifier",
    "FirstName":               "First Name",
    "LastName":                "Last Name",
    "EpisodeEndDate":          "Current Episode End Date",
    "AgencyName":              "Agency Name",
    "ProgramName":             "Program Name",
    "ProjectTypeCode":         "Project Type",
    "ProjectStartDate":        "Enrollment Start Date",
}
no_ce_df["_end_sort"] = pd.to_datetime(no_ce_df["EpisodeEndDate"], errors="coerce")
no_ce_df = no_ce_df.sort_values("_end_sort", ascending=False, na_position="first").drop(columns="_end_sort")
no_ce_inactive_excel_rows = [
    i + 2
    for i, v in enumerate(no_ce_df["EpisodeOutflowType"] == "Inactive")
    if v
]
tab_no_ce = select_columns(no_ce_df, NO_CE_COL_MAP)


# ── Tab 4: Newly Assessed ──────────────────────────────────────────────────────────────
NEWLY_ASSESSED_COL_MAP = {
    "client_assessments.assessment_date":                              "Assessment Date",
    "agencies.name":                                                   "Assessing Agency",
    "screens.name":                                                    "Assessment Name",
    "client_assessments.ref_user":                                     "User Creating",
    "clients.unique_identifier":                                       "Unique Identifier",
    "client.full_name":                                                "Client Full Name",
    "client.ssn3":                                                     "SSN - Last 4",
    "client.birth_date":                                               "Date of Birth",
    "HouseholdTotalIncome":                                            "Total Household Income",
    "entry_custom.c_kc_is_household_chronically_homeless":             "Chronic Homeless Questions Complete",
    "static_demographics.gender_text":                                 "Gender",
    "client_assessments.ssvf_targeting_field_12":                      "Registered Sex Offender",
    "entry_screen.added_date":                                         "Entry Screen Added Date",
    "chronically_homeless_households.is_chronic_homeless_household":   "Chronically Homeless at PIT - Household",
    "entry_screen.health_chronic":                                     "Chronic Health",
    "entry_screen.health_chronic_longterm":                            "Chronic Health Longterm",
    "entry_screen.health_dev_disability":                              "Developmental",
    "DisablingConditionAnytime":                                       "Disabling Condition",
    "entry_screen.health_mental":                                      "Mental Health",
    "entry_screen.health_mental_longterm":                             "Mental Health Longterm",
    "entry_screen.health_phys_disability":                             "Physical",
    "entry_screen.health_phys_disability_longterm":                    "Physical Longterm",
    "entry_screen.health_substance_abuse":                             "Substance Use Disorder",
    "entry_screen.health_substance_abuse_longterm":                    "Substance Use Longterm",
    "entry_screen.health_insurance":                                   "Covered by Health Insurance",
    "entry_screen.income_vet_disability_is":                           "Veteran Disability",
    "entry_screen.income_vet_pension_is":                              "VA Non-Service Connected Disability Pension",
    "entry_screen.income_earned":                                      "Earned Income Amount",
    "entry_screen.benefits_noncash":                                   "Non-Cash Benefit from Any Source",
}
newly_assessed_raw = newly_assessed_raw.merge(
    hh_income.rename(columns={"ClientUniqueIdentifier": "clients.unique_identifier"}),
    on="clients.unique_identifier", how="left"
)
tab_newly_assessed = select_columns(newly_assessed_raw, NEWLY_ASSESSED_COL_MAP)


# ── Tab 5: Flagged ─────────────────────────────────────────────────────────────────────
# Compute total household income from parquet: sum MonthlyCashIncome for all open
# enrollment household members of each flagged client.
flagged_raw = flagged_raw.merge(
    hh_income.rename(columns={"ClientUniqueIdentifier": "clients.unique_identifier"}),
    on="clients.unique_identifier", how="left"
)

# Column order matches spec exactly (VBNL Workbook Specs.docx → Flagged tab)
FLAGGED_COL_MAP = {
    "client_assessments.assessment_date":                              "Assessment Date",
    "agencies.name":                                                   "Assessing Agency",
    "screens.name":                                                    "Assessment Name",
    "client_assessments.ref_user":                                     "User Creating",
    "clients.unique_identifier":                                       "Unique Identifier",
    "client.full_name":                                                "Client Full Name",
    "client.ssn3":                                                     "SSN - Last 4",
    "client.birth_date":                                               "Date of Birth",
    "HouseholdTotalIncome":                                            "Total Household Income",
    "entry_custom.c_kc_is_household_chronically_homeless":             "Chronic Homeless Questions Complete",
    "static_demographics.gender_text":                                 "Gender",
    "client_assessment_custom.c_reg_sex_offender":                     "Registered Sex Offender",
    "entry_screen.added_date":                                         "Entry Screen Added Date",
    "chronically_homeless_households.is_chronic_homeless_household":   "Chronically Homeless at PIT - Household",
    "entry_screen.health_chronic":                                     "Chronic Health",
    "entry_screen.health_chronic_longterm":                            "Chronic Health Longterm",
    "entry_screen.health_dev_disability":                              "Developmental",
    "DisablingConditionAnytime":                                       "Disabling Condition",
    "entry_screen.health_mental":                                      "Mental Health",
    "entry_screen.health_mental_longterm":                             "Mental Health Longterm",
    "entry_screen.health_phys_disability":                             "Physical",
    "entry_screen.health_phys_disability_longterm":                    "Physical Longterm",
    "entry_screen.health_substance_abuse":                             "Substance Use Disorder",
    "entry_screen.health_substance_abuse_longterm":                    "Substance Use Longterm",
    "entry_screen.health_insurance":                                   "Covered by Health Insurance",
    "entry_screen.income_vet_disability_is":                           "Veteran Disability",
    "entry_screen.income_vet_pension_is":                              "VA Non-Service Connected Disability Pension",
    "entry_screen.income_earned":                                      "Earned Income Amount",
    "entry_screen.benefits_noncash":                                   "Non-Cash Benefit from Any Source",
    "client_assessment_custom.c_kc_volt_notes":                        "VBNL Notes",
    "client_assessment_custom.c_kc_volt_review_flag":                  "Flag for Re-Review",
}
tab_flagged = select_columns(flagged_raw, FLAGGED_COL_MAP)

# Write Workbook to Azure Storage

In [ ]:
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

# Brand colors — replace with your organization's palette
PRIMARY_COLOR = "172B69"   # Navy — primary header fill
ACCENT_COLOR  = "1B8477"   # Teal — accent (available for secondary headers)
WHITE         = "FFFFFF"

CURRENCY_FMT = '"$"#,##0.00'   # applied to any column with "Income" in the header

def apply_brand_style(ws):
    """Apply header branding, currency formatting, and basic layout to a worksheet."""
    header_fill = PatternFill("solid", fgColor=PRIMARY_COLOR)
    header_font = Font(bold=True, color=WHITE, name="Arial Nova", size=11)

    # Collect header names for currency detection
    headers = [cell.value for cell in ws[1]]

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="left", vertical="center", wrap_text=False)

    ws.row_dimensions[1].height = 20
    ws.freeze_panes = "A2"

    # Apply currency format to any column whose header contains "Income"
    for col_idx, header in enumerate(headers, start=1):
        if header and "income" in str(header).lower():
            col_letter = get_column_letter(col_idx)
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    cell.number_format = CURRENCY_FMT

    # Auto-fit column widths (capped at 60)
    for col_idx, col_cells in enumerate(ws.columns, start=1):
        max_len = max(
            (len(str(c.value)) if c.value is not None else 0) for c in col_cells
        )
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 4, 60)


output_filename = f"VBNL_{run_date}.xlsx"

# Build the workbook in memory only — the unencrypted version must never touch disk.
workbook_buffer = io.BytesIO()

with pd.ExcelWriter(workbook_buffer, engine="openpyxl") as writer:
    tab_active.to_excel(writer, sheet_name="Active", index=False)
    tab_inactive.to_excel(writer, sheet_name="Recently Inactive", index=False)
    tab_no_ce.to_excel(writer, sheet_name="No CE Engagement", index=False)
    tab_newly_assessed.to_excel(writer, sheet_name="Newly Assessed", index=False)
    tab_flagged.to_excel(writer, sheet_name="Flagged", index=False)

    wb = writer.book
    for ws in wb.worksheets:
        apply_brand_style(ws)

    # ── Grey shading for recently-inactive rows on the Active sheet ───────────────────
    ws_active = wb["Active"]
    inactive_fill = PatternFill("solid", fgColor="E0E0E0")
    for row_num in inactive_excel_rows:
        for cell in ws_active[row_num]:
            cell.fill = inactive_fill

    # ── Grey shading for recently-inactive rows on the No CE Engagement sheet ────────
    ws_no_ce = wb["No CE Engagement"]
    for row_num in no_ce_inactive_excel_rows:
        for cell in ws_no_ce[row_num]:
            cell.fill = inactive_fill

    # ── Cover tab ────────────────────────────────────────────────────────────────────
    ws_cover = wb.create_sheet("Summary", 0)
    ws_cover.column_dimensions["A"].width = 53.16
    ws_cover.column_dimensions["B"].width = 17.83
    ws_cover.row_dimensions[1].height = 43
    ws_cover.row_dimensions[2].height = 43

    teal_fill   = PatternFill("solid", fgColor="B9E9E3")
    yellow_fill = PatternFill("solid", fgColor="FFF8C7")
    label_font  = Font(name="Arial Nova", size=24, bold=False)
    count_font  = Font(name="Arial Nova", size=24, bold=True)

    ws_cover["A1"].value     = "Actively Homeless Veterans:"
    ws_cover["A1"].font      = label_font
    ws_cover["A1"].fill      = teal_fill
    ws_cover["A1"].alignment = Alignment(vertical="center")
    ws_cover["B1"].value     = n_active_vets
    ws_cover["B1"].font      = count_font
    ws_cover["B1"].fill      = teal_fill
    ws_cover["B1"].alignment = Alignment(horizontal="center", vertical="center")

    ws_cover["A2"].value     = "Recently Inactive Veterans:"
    ws_cover["A2"].font      = label_font
    ws_cover["A2"].fill      = yellow_fill
    ws_cover["A2"].alignment = Alignment(vertical="center")
    ws_cover["B2"].value     = n_recently_inactive_vets
    ws_cover["B2"].font      = count_font
    ws_cover["B2"].fill      = yellow_fill
    ws_cover["B2"].alignment = Alignment(horizontal="center", vertical="center")

print(f"Workbook built in memory (not yet saved to disk): {output_filename}")
print(f"  Active:                  {len(tab_active):>4} rows")
print(f"  Inactive Since Last BNL: {len(tab_inactive):>4} rows")
print(f"  No CE Engagement:        {len(tab_no_ce):>4} rows")
print(f"  Newly Assessed:          {len(tab_newly_assessed):>4} rows")
print(f"  Flagged:                 {len(tab_flagged):>4} rows")

In [ ]:
# ── Password-protect the workbook ─────────────────────────────────────────────────────
# Requires: pip install msoffcrypto-tool
import msoffcrypto

# # Development: prompts for the password without echoing it
import getpass
password = getpass.getpass("Workbook password: ")

# Production: fetch from Key Vault
# password = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]', '[[WORKBOOK_PASSWORD_SECRET]]')

# Encrypt directly from the in-memory buffer — only the encrypted bytes are ever written to disk.
workbook_buffer.seek(0)
encrypted = io.BytesIO()
msoffcrypto.OfficeFile(workbook_buffer).encrypt(password, encrypted)

with open(output_filename, "wb") as f:
    f.write(encrypted.getvalue())

print(f"Password-protected: {output_filename}")

in development, enter az login to terminal

In [ ]:
# Development -- comment out in production
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

_kv = SecretClient(
    vault_url="https://[[AZURE_KEY_VAULT_NAME]].vault.azure.net/",
    credential=DefaultAzureCredential()
)
client_id     = _kv.get_secret("[[SHAREPOINT_CLIENT_ID_SECRET]]").value
client_secret = _kv.get_secret("[[SHAREPOINT_CLIENT_SECRET_SECRET]]").value
tenant_id     = _kv.get_secret("[[SHAREPOINT_TENANT_ID_SECRET]]").value

# # Production -- comment out in development
# client_id     = TokenLibrary.getSecretWithLS("[[AZURE_KEY_VAULT_NAME]]", "[[SHAREPOINT_CLIENT_ID_SECRET]]")
# client_secret = TokenLibrary.getSecretWithLS("[[AZURE_KEY_VAULT_NAME]]", "[[SHAREPOINT_CLIENT_SECRET_SECRET]]")
# tenant_id     = TokenLibrary.getSecretWithLS("[[AZURE_KEY_VAULT_NAME]]", "[[SHAREPOINT_TENANT_ID_SECRET]]")

# Shared -- runs in both envs
import msal
SP_HOSTNAME  = "[[SHAREPOINT_HOSTNAME]]"
SP_SITE_PATH = "[[SHAREPOINT_SITE_PATH]]"
SP_FOLDER    = "[[SHAREPOINT_FOLDER_PATH]]"

app = msal.ConfidentialClientApplication(
    client_id, client_credential=client_secret,
    authority=f"https://login.microsoftonline.com/{tenant_id}",
)
result = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
if "access_token" not in result:
    raise Exception(f"SharePoint auth failed: {result.get('error_description')}")
sp_headers = {"Authorization": f"Bearer {result['access_token']}"}

site_resp = requests.get(
    f"https://graph.microsoft.com/v1.0/sites/{SP_HOSTNAME}:{SP_SITE_PATH}",
    headers=sp_headers,
)
site_resp.raise_for_status()
sp_site_id = site_resp.json()["id"]

In [ ]:
# ── Upload password-protected workbook to SharePoint ──────────────────────────
with open(output_filename, "rb") as f:
    file_bytes = f.read()

upload_url = (
    f"https://graph.microsoft.com/v1.0/sites/{sp_site_id}"
    f"/drive/root:/{SP_FOLDER}/{output_filename}:/content"
)

response = requests.put(
    upload_url,
    headers={**sp_headers, "Content-Type": "application/octet-stream"},
    data=file_bytes,
)
response.raise_for_status()
print(f"Uploaded to SharePoint: {SP_FOLDER}/{output_filename}")